# 03 — Sequences and Batching

## Goal

A language model is trained to predict the next token from previous
tokens.

After tokenization, text is represented as a sequence of token IDs:

$$
x_1, x_2, x_3, \ldots, x_T.
$$

The training objective is to model

$$
p(x_t \mid x_1, x_2, \ldots, x_{t-1}).
$$

This means that a single token sequence can generate many supervised
training examples without requiring manually labeled data.

In this lesson, we will build these training examples from scratch and
then organize them into batches suitable for a language model.

In [1]:
token_ids: list[int] = [10, 11, 12, 13, 14, 15]

print(token_ids)

[10, 11, 12, 13, 14, 15]


## 1. Next-Token Prediction

Given a token sequence

$$
[x_1, x_2, x_3, \ldots, x_T],
$$

the model learns relationships such as

$$
x_1 \rightarrow x_2,
$$

$$
(x_1, x_2) \rightarrow x_3,
$$

$$
(x_1, x_2, x_3) \rightarrow x_4,
$$

and so on.

A convenient way to represent this learning problem is to create two
shifted sequences:

$$
X = [x_1, x_2, \ldots, x_{T-1}]
$$

and

$$
Y = [x_2, x_3, \ldots, x_T].
$$

At every position, the token in `Y` is the next-token target for the
corresponding position in `X`.

In [2]:
# construct "shifts" for the next token pred
inputs: list[int] = token_ids[:-1]
targets: list[int] = token_ids[1:]

print("inputs: ", inputs)
print("targets:", targets)

inputs:  [10, 11, 12, 13, 14]
targets: [11, 12, 13, 14, 15]


## 2. Context Length

A language model does not necessarily process the entire corpus at once.

Instead, training uses fixed-length token blocks.

Let the context length be

$$
T.
$$

A training block contains $T$ input tokens and $T$ corresponding
next-token targets.

To construct such a block, we actually need

$$
T + 1
$$

consecutive tokens from the corpus.

For example, with a context length of

$$
T = 4,
$$

the token sequence

<pre>
10  11  12  13  14
</pre>

produces

<pre>
input:   10  11  12  13
target:  11  12  13  14
</pre>

The extra token is needed because every input position requires a
next-token target.

In [3]:
def make_training_block(
    tokens: list[int], start: int, context_length: int
) -> tuple[list[int], list[int]]:
    block: list[int] = tokens[start : start + context_length + 1]
    inputs: list[int] = block[:-1]
    targets: list[int] = block[1:]

    return inputs, targets


tokens: list[int] = [10, 11, 12, 13, 14, 15, 16, 17]

x, y = make_training_block(
    tokens=tokens,
    start=1,
    context_length=4,
)

print("x:", x)
print("y:", y)

x: [11, 12, 13, 14]
y: [12, 13, 14, 15]


## 3. Sliding Windows

A long token sequence can generate many training blocks.

Suppose the corpus contains $N$ tokens and the context length is $T$.

Each training block requires

$$
T + 1
$$

consecutive tokens:

- the first $T$ tokens become the input,
- the last $T$ tokens become the targets after shifting by one position.

A starting position `start` is valid only if

$$
start + T < N.
$$

Equivalently,

$$
start \leq N - T - 1.
$$

Therefore, with stride 1, the number of possible training blocks is

$$
N - T.
$$

A sliding window can move through the corpus and create overlapping
training examples.

In [4]:
TrainingExample = tuple[list[int], list[int]]


def make_training_examples(
    tokens: list[int], context_length: int, stride: int = 1
) -> list[TrainingExample]:
    examples: list[TrainingExample] = []
    max_start: int = len(tokens) - context_length
    for start in range(0, max_start, stride):
        inputs, targets = make_training_block(
            tokens=tokens, start=start, context_length=context_length
        )

        examples.append((inputs, targets))

    return examples


tokens: list[int] = [10, 11, 12, 13, 14, 15, 16, 17]

examples: list[TrainingExample] = make_training_examples(
    tokens=tokens,
    context_length=4,
)

for index, (inputs, targets) in enumerate(examples):
    print(f"example {index}")
    print("  input: ", inputs)
    print("  target:", targets)

example 0
  input:  [10, 11, 12, 13]
  target: [11, 12, 13, 14]
example 1
  input:  [11, 12, 13, 14]
  target: [12, 13, 14, 15]
example 2
  input:  [12, 13, 14, 15]
  target: [13, 14, 15, 16]
example 3
  input:  [13, 14, 15, 16]
  target: [14, 15, 16, 17]


### Stride

The `stride` determines how far the window moves between consecutive
training examples.

With

$$
stride = 1,
$$

neighboring examples overlap heavily.

With

$$
stride = T,
$$

**the input blocks do not overlap**.

A smaller stride produces more training examples from the same corpus,
but also introduces more redundancy between neighboring examples.

In [5]:
examples_stride_2 = make_training_examples(
    tokens=tokens,
    context_length=4,
    stride=2,
)

for inputs, targets in examples_stride_2:
    print(inputs, "->", targets)

[10, 11, 12, 13] -> [11, 12, 13, 14]
[12, 13, 14, 15] -> [13, 14, 15, 16]


## 4. Batching

A single training sequence has shape

$$
(T),
$$

where $T$ is the context length.

Training usually processes several sequences in parallel.

If the batch contains $B$ sequences, the input tensor has shape

$$
(B, T).
$$

The target tensor has the same shape:

$$
(B, T).
$$

The dimensions mean:

<pre>
dim 0 → batch examples
dim 1 → token positions within each example
</pre>

In [6]:
import torch

batch_exmaples: list[TrainingExample] = examples[:3]

input_batch = torch.tensor(
    [inputs for inputs, _ in batch_exmaples], dtype=torch.long
)

target_batch = torch.tensor(
    [targets for _, targets in batch_exmaples], dtype=torch.long
)

print("inputs:")
print(input_batch)

print("\ntargets:")
print(target_batch)

print("\ninput shape:", input_batch.shape)
print("target shape:", target_batch.shape)

inputs:
tensor([[10, 11, 12, 13],
        [11, 12, 13, 14],
        [12, 13, 14, 15]])

targets:
tensor([[11, 12, 13, 14],
        [12, 13, 14, 15],
        [13, 14, 15, 16]])

input shape: torch.Size([3, 4])
target shape: torch.Size([3, 4])


## 5. Train and Validation Splits

A model should be evaluated on data that was not used to update its
parameters.

We therefore divide the token stream into separate subsets:

<pre>
full token stream
       ↓
train tokens | validation tokens
</pre>

Training batches are sampled only from the training portion.

Validation batches are sampled only from the validation portion.

The split should happen **before** creating overlapping training
windows.

If overlapping windows are created first and then randomly assigned to
training and validation sets, nearly identical sequences may appear in
both splits.

For example:

<pre>
window A: 10 11 12 13 14
window B:    11 12 13 14 15
</pre>

If `A` is placed in the training set and `B` in the validation set,
most of the validation example has already been seen during training.

This is a form of data leakage.

In [7]:
def split_tokens(
    tokens: list[int], train_fraction: float
) -> tuple[list[int], list[int]]:
    if not 0.0 < train_fraction < 1.0:
        raise ValueError("`train_fraction` must be between 0 and 1")
    split_index: int = int(len(tokens) * train_fraction)

    train_tokens: list[int] = tokens[:split_index]
    validation_tokens: list[int] = tokens[split_index:]

    return train_tokens, validation_tokens


# test
tokens: list[int] = list(range(20))

train_tokens, validation_tokens = split_tokens(
    tokens=tokens,
    train_fraction=0.8,
)

print("train:", train_tokens)
print("validation:", validation_tokens)

train: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
validation: [16, 17, 18, 19]


## 6. Random Batch Sampling

It is not necessary to materialize every possible overlapping training
window in memory.

Instead, each training step can randomly sample valid starting
positions from the token stream.

For a token stream of length $N$ and context length $T$, a valid start
position must satisfy

$$
start + T < N.
$$

For every sampled starting position $s$, we construct

$$
X =
[x_s, x_{s+1}, \ldots, x_{s+T-1}]
$$

and

$$
Y =
[x_{s+1}, x_{s+2}, \ldots, x_{s+T}].
$$

Sampling several starting positions produces a batch of shape

$$
(B,T).
$$

The token order inside each sequence remains unchanged. Only the
starting positions of the sampled windows are randomized.

In [10]:
def get_batch(
    tokens: list[int], batch_size: int, context_length: int
) -> tuple[torch.Tensor, torch.Tensor]:
    max_start: int = len(tokens) - context_length

    if max_start <= 0:
        raise ValueError(
            "Token sequence is too short for the requested context length"
        )

    start_positions: torch.Tensor = torch.randint(
        low=0, high=max_start, size=(batch_size,)
    )

    inputs: list[list[int]] = []
    targets: list[list[int]] = []

    for start_tensor in start_positions:
        start: int = int(start_tensor.item())

        inputs.append(tokens[start : start + context_length])
        targets.append(tokens[start + 1 : start + context_length + 1])

    input_batch = torch.tensor(inputs, dtype=torch.long)
    target_batch = torch.tensor(targets, dtype=torch.long)

    return input_batch, target_batch


# test

tokens: list[int] = list(range(100))

input_batch, target_batch = get_batch(
    tokens=tokens,
    batch_size=4,
    context_length=8,
)

print("inputs:")
print(input_batch)

print("\ntargets:")
print(target_batch)

print("\ninput shape:", input_batch.shape)
print("target shape:", target_batch.shape)
print("dtype:", input_batch.dtype)

inputs:
tensor([[74, 75, 76, 77, 78, 79, 80, 81],
        [65, 66, 67, 68, 69, 70, 71, 72],
        [ 0,  1,  2,  3,  4,  5,  6,  7],
        [91, 92, 93, 94, 95, 96, 97, 98]])

targets:
tensor([[75, 76, 77, 78, 79, 80, 81, 82],
        [66, 67, 68, 69, 70, 71, 72, 73],
        [ 1,  2,  3,  4,  5,  6,  7,  8],
        [92, 93, 94, 95, 96, 97, 98, 99]])

input shape: torch.Size([4, 8])
target shape: torch.Size([4, 8])
dtype: torch.int64


## 7. Fixed-Length Blocks and Padding

Our current batching strategy samples fixed-length windows from a long
token stream.

Every example contains exactly

$$
T
$$

input tokens and exactly

$$
T
$$

target tokens.

Therefore, a batch can be stacked directly into a tensor of shape

$$
(B, T)
$$

without padding.

For example:

<pre>
example 0: 10 11 12 13
example 1: 24 25 26 27
example 2: 41 42 43 44
</pre>

already forms a rectangular tensor.

Padding becomes necessary when examples have different sequence lengths.

## 8. Variable-Length Sequences

A tensor batch must have a rectangular shape.

If individual sequences have different lengths,

<pre>
10 11 12 13
20 21
30 31 32
</pre>

they cannot be directly represented as a single `(B, T)` tensor.

One solution is to introduce a reserved padding token.

If `<pad>` has token ID `0`, the sequences can become:

<pre>
10 11 12 13
20 21  0  0
30 31 32  0
</pre>

The padding token does not represent ordinary text. It exists only to
make the batch rectangular.

In [12]:
def pad_sequences(
    sequences: list[list[int]], pad_id: int
) -> tuple[torch.Tensor, torch.Tensor]:
    if not sequences:
        raise ValueError("Cannot pad an empty batch.")
    max_length: int = max(len(sequence) for sequence in sequences)

    padded: list[list[int]] = []
    masks: list[list[bool]] = []

    for sequence in sequences:
        padding_length: int = max_length - len(sequence)

        padded_sequence: list[int] = sequence + [pad_id] * padding_length

        mask: list[bool] = [True] * len(sequence) + [False] * padding_length

        padded.append(padded_sequence)
        masks.append(mask)

    return torch.tensor(padded, dtype=torch.long), torch.tensor(
        masks, dtype=torch.bool
    )


sequences: list[list[int]] = [
    [10, 11, 12, 13],
    [20, 21],
    [30, 31, 32],
]

padded_batch, padding_mask = pad_sequences(
    sequences=sequences,
    pad_id=0,
)

print("padded:")
print(padded_batch)

print("\nmask:")
print(padding_mask)

print("\nshapes:")
print(padded_batch.shape)
print(padding_mask.shape)

padded:
tensor([[10, 11, 12, 13],
        [20, 21,  0,  0],
        [30, 31, 32,  0]])

mask:
tensor([[ True,  True,  True,  True],
        [ True,  True, False, False],
        [ True,  True,  True, False]])

shapes:
torch.Size([3, 4])
torch.Size([3, 4])


## 9. Three Different Kinds of Masks

Several different masks may appear in a decoder-only language model.
They solve different problems.

### Causal mask

A causal mask prevents a token from attending to future positions.

For a sequence of length four:

<pre>
        key
        0 1 2 3

q = 0   ✓ ✗ ✗ ✗
q = 1   ✓ ✓ ✗ ✗
q = 2   ✓ ✓ ✓ ✗
q = 3   ✓ ✓ ✓ ✓
</pre>

This mask expresses the autoregressive constraint.

### Padding mask

A padding mask identifies positions that were inserted only to make
variable-length sequences rectangular.

<pre>
token:  20 21 PAD PAD
mask:    ✓  ✓  ✗   ✗
</pre>

The model should not treat padding as ordinary content.

### Loss mask

A loss mask specifies which target positions contribute to the training
objective.

Targets corresponding to padding should usually not contribute to the
language-model loss.

These masks may have related information, but they are conceptually
different:

<pre>
causal mask   → which positions may attend to which positions
padding mask  → which positions contain real input tokens
loss mask     → which prediction positions contribute to the loss
</pre>

In [14]:
PAD_ID: int = 0
BOS_ID: int = 1
EOS_ID: int = 2

sequences: list[list[int]] = [
    [BOS_ID, 10, 11, 12, EOS_ID],
    [BOS_ID, 20, EOS_ID],
    [BOS_ID, 30, 31, EOS_ID],
]

padded, token_mask = pad_sequences(
    sequences,
    pad_id=PAD_ID,
)

print(padded)


inputs: torch.Tensor = padded[:, :-1]
targets: torch.Tensor = padded[:, 1:]

print(inputs)
print(targets)

tensor([[ 1, 10, 11, 12,  2],
        [ 1, 20,  2,  0,  0],
        [ 1, 30, 31,  2,  0]])
tensor([[ 1, 10, 11, 12],
        [ 1, 20,  2,  0],
        [ 1, 30, 31,  2]])
tensor([[10, 11, 12,  2],
        [20,  2,  0,  0],
        [30, 31,  2,  0]])


In [ ]:
# <EOS> should count as the loss calculation, because the model should learn when to stop
loss_mask: torch.Tensor = targets != PAD_ID
print("inputs:")
print(inputs)

print("\ntargets:")
print(targets)

print("\nloss mask:")
print(loss_mask)


inputs:
tensor([[ 1, 10, 11, 12],
        [ 1, 20,  2,  0],
        [ 1, 30, 31,  2]])

targets:
tensor([[10, 11, 12,  2],
        [20,  2,  0,  0],
        [30, 31,  2,  0]])

loss mask:
tensor([[ True,  True,  True,  True],
        [ True,  True, False, False],
        [ True,  True,  True, False]])


## 10. Document Boundaries

A training corpus may contain multiple independent documents.

Simply concatenating them creates transitions between documents:

<pre>
document A | document B
</pre>

Without an explicit boundary marker, the model may be trained to predict
the beginning of document B directly from the end of document A.

A common solution is to insert an end-of-sequence token:

<pre>
document A <EOS> document B <EOS> document C <EOS>
</pre>

The model can then learn that `<EOS>` represents a meaningful sequence
boundary.

Whether training examples are allowed to cross document boundaries is a
separate design decision.

The important point is that document boundaries should be represented
deliberately rather than appearing accidentally.

In [16]:
text: str = "the small language model learns to predict the next token."

vocab: list[str] = sorted(set(text))

char_to_id: dict[str, int] = {char: index for index, char in enumerate(vocab)}


def encode_text(
    text: str,
    token_to_id: dict[str, int],
) -> list[int]:
    return [token_to_id[char] for char in text]


token_ids: list[int] = encode_text(
    text,
    char_to_id,
)

print("text length:", len(text))
print("token count:", len(token_ids))
print("first token IDs:", token_ids[:20])

text length: 58
token count: 58
first token IDs: [17, 7, 5, 0, 16, 11, 2, 10, 10, 0, 10, 2, 12, 6, 18, 2, 6, 5, 0, 11]


In [17]:
input_batch, target_batch = get_batch(
    tokens=token_ids,
    batch_size=4,
    context_length=8,
)

print(input_batch)
print(target_batch)

tensor([[ 0, 17, 13,  0, 14, 15,  5,  4],
        [12, 16,  0, 17, 13,  0, 14, 15],
        [ 0, 11, 13,  4,  5, 10,  0, 10],
        [ 2,  6,  5,  0, 11, 13,  4,  5]])
tensor([[17, 13,  0, 14, 15,  5,  4,  8],
        [16,  0, 17, 13,  0, 14, 15,  5],
        [11, 13,  4,  5, 10,  0, 10,  5],
        [ 6,  5,  0, 11, 13,  4,  5, 10]])


In [18]:
def decode_ids(
    token_ids: list[int],
    id_to_token: dict[int, str],
) -> str:
    return "".join(id_to_token[token_id] for token_id in token_ids)


id_to_char: dict[int, str] = {index: char for index, char in enumerate(vocab)}

In [19]:
first_input: list[int] = input_batch[0].tolist()

first_target: list[int] = target_batch[0].tolist()

print(
    "input:",
    repr(
        decode_ids(
            first_input,
            id_to_char,
        )
    ),
)

print(
    "target:",
    repr(
        decode_ids(
            first_target,
            id_to_char,
        )
    ),
)

input: ' to pred'
target: 'to predi'


## Takeaways

- Language modeling converts an unlabeled token stream into next-token
  prediction examples.
- Inputs and targets are the same sequence shifted by one position.
- A context length of $T$ requires $T+1$ source tokens to construct one
  training example.
- A single `(B, T)` batch contains $B \times T$ next-token prediction
  targets.
- Train and validation token streams should be separated before creating
  highly overlapping windows.
- Random batch sampling changes the selected windows but preserves token
  order inside each window.
- Fixed-length blocks do not require padding.
- Variable-length batches may require a dedicated padding token.
- Causal masks, padding masks, and loss masks solve different problems.
- `<EOS>` represents a meaningful sequence boundary, while `<PAD>` is a
  batching artifact.
- Token IDs remain integer indices until an embedding layer converts
  them into continuous vectors.